In [ ]:
# ---------------------------------------------------------
# PREDICT INSULIN REQUIREMENTS
# MULTIPLE MODEL COMPARISON
# ---------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------

df = pd.read_excel("Team6_DataDynamos_Python-Hackathon_MAY2026_V2.xlsx")

# ---------------------------------------------------------
# CREATE FUTURE INSULIN TARGET
# ---------------------------------------------------------

# Predict next insulin dosage
df['future_insulin'] = df['bolus_volume_delivered'].shift(-1)

# Remove missing rows
df = df.dropna()

# ---------------------------------------------------------
# FEATURES
# ---------------------------------------------------------

features = [

    'glucose',
    'carb_input',
    'calories',
    'steps',
    'average_sleep_duration_(hrs)',
    'heart_rate',
    'basal_rate'

]

X = df[features]

# Target Variable
y = df['future_insulin']

# ---------------------------------------------------------
# TRAIN TEST SPLIT
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)

# ---------------------------------------------------------
# DEFINE MODELS
# ---------------------------------------------------------

models = {

    "Linear Regression":
        LinearRegression(),

    "Decision Tree":
        DecisionTreeRegressor(random_state=42),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        ),

    "KNN":
        KNeighborsRegressor(),

    "SVR":
        SVR()

}

# ---------------------------------------------------------
# TRAIN & EVALUATE MODELS
# ---------------------------------------------------------

results = []

for name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation
    mae = mean_absolute_error(y_test, y_pred)

    r2 = r2_score(y_test, y_pred)

    results.append([name, mae, r2, model])

    print("\n--------------------------------")
    print("Model:", name)

    print("MAE:", round(mae, 4))

    print("R2 Score:", round(r2, 4))

# ---------------------------------------------------------
# SELECT BEST MODEL
# ---------------------------------------------------------

# Best model = Highest R2 Score
best_model_info = max(results, key=lambda x: x[2])

best_name = best_model_info[0]
best_mae = best_model_info[1]
best_r2 = best_model_info[2]
best_model = best_model_info[3]

print("\n================================================")
print("BEST MODEL")
print("================================================")

print("Best Model:", best_name)

print("Best MAE:", round(best_mae, 4))

print("Best R2 Score:", round(best_r2, 4))

# ---------------------------------------------------------
# TEST BEST MODEL
# ---------------------------------------------------------

y_pred_best = best_model.predict(X_test)

# Create comparison table
comparison = pd.DataFrame({

    'Actual Insulin': y_test.values,
    'Predicted Insulin': y_pred_best

})

print("\n================================================")
print("ACTUAL VS PREDICTED INSULIN")
print("================================================")

print(comparison.head(20))

# ---------------------------------------------------------
# TEST SAMPLE PATIENT DATA
# ---------------------------------------------------------

sample_patient = pd.DataFrame({

    'glucose': [240],
    'carb_input': [85],
    'calories': [700],
    'steps': [1200],
    'average_sleep_duration_(hrs)': [5],
    'heart_rate': [96],
    'basal_rate': [1.1]

})

# Predict insulin requirement
predicted_insulin = best_model.predict(sample_patient)

print("\n================================================")
print("SAMPLE PATIENT TEST")
print("================================================")

print("Predicted Future Insulin Requirement:",
      round(predicted_insulin[0], 2))

# ---------------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------------

# Only works for tree-based models
if best_name == "Random Forest":

    importance = pd.DataFrame({

        'Feature': features,
        'Importance': best_model.feature_importances_

    })

    importance = importance.sort_values(
        by='Importance',
        ascending=False
    )

    print("\n================================================")
    print("FEATURE IMPORTANCE")
    print("================================================")

    print(importance)


--------------------------------
Model: Linear Regression
MAE: 0.1268
R2 Score: 0.0155
